# Первый Colab run: шумоподавление русской речи

Этот ноутбук делает первый короткий end-to-end прогон:

- работает из текущей рабочей папки проекта в VSCode/Colab runtime
- подготавливает маленький набор русской речи
- генерирует маленький synthetic noisy dataset
- обучает tiny STFT-модель
- сохраняет checkpoint **после каждой эпохи локально** в `outputs/colab_first_result`
- сравнивает tiny model с `identity` baseline на `SI-SDR`

Важно:

- это **не** финальная модель, а быстрый диагностический прогон;
- notebook предполагает, что вы открыли его из корня проекта или что корень проекта доступен рядом с notebook.


## 1. Настраиваем рабочую директорию

Для VSCode Colab extension ожидаемый сценарий такой:

- вы ведете проект в git;
- указываете `REPO_URL` в первой ячейке;
- если `/content/project` еще нет, первая ячейка сама выполнит `git clone`;
- если проект уже есть, первая ячейка просто перейдет в его корень.

Google Drive здесь **не используется**.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

# ВАЖНО: замените на URL вашего git-репозитория.
# Для публичного репозитория подойдет HTTPS URL.
# Пример: REPO_URL = 'https://github.com/your-name/noise-suppression-mvp.git'
REPO_URL = 'https://github.com/PushinMax/noise-suppression-mvp.git'


def run(command: str) -> None:
    print(f"\n$ {command}", flush=True)
    subprocess.run(command, shell=True, check=True, executable='/bin/bash')


def reset_paths(*paths: str) -> None:
    for raw_path in paths:
        path = Path(raw_path)
        if path.is_dir():
            shutil.rmtree(path)
            print(f'Удалена папка: {path}')
        elif path.exists():
            path.unlink()
            print(f'Удален файл: {path}')


def read_manifest(path: str) -> list[dict]:
    manifest_path = Path(path)
    if not manifest_path.exists():
        raise FileNotFoundError(f'Не найден manifest: {manifest_path}')
    return [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]


def assert_nonempty_manifest(path: str) -> list[dict]:
    rows = read_manifest(path)
    if not rows:
        raise RuntimeError(f'Manifest пуст: {path}. Остановились, чтобы не получить вторичные ошибки ниже.')
    print(f'{path}: rows = {len(rows)}')
    return rows


def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src/noise_suppression').exists()


def find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if is_project_root(candidate):
            return candidate
    fallback = Path('/content/project')
    if is_project_root(fallback):
        return fallback
    return None


PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    if not REPO_URL:
        raise FileNotFoundError(
            'Не найден корень проекта. Укажите REPO_URL в первой ячейке notebook, '
            'например https://github.com/<user>/<repo>.git, и перезапустите ячейку.'
        )
    PROJECT_ROOT = Path('/content/project')
    if PROJECT_ROOT.exists():
        raise FileExistsError(
            f'{PROJECT_ROOT} уже существует, но это не похоже на корень проекта. '
            'Удалите папку или проверьте путь.'
        )
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)

RUN_NAME = 'first_colab_result'
RUNTIME_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / RUN_NAME

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUNTIME_OUTPUT_DIR =', RUNTIME_OUTPUT_DIR)
print('pyproject exists =', (PROJECT_ROOT / 'pyproject.toml').exists())
print('package exists =', (PROJECT_ROOT / 'src/noise_suppression').exists())


## 2. Устанавливаем зависимости

Здесь мы ставим:

- `uv`
- train stack
- data stack для подготовки маленького датасета


In [ ]:
run('pip -q install uv')
run('uv sync --extra train --extra data --extra dev')
run('uv run noise-suppression env check')


## 3. Готовим маленькие данные для первого результата

В этом прогоне берем:

- небольшой кусок `google/fleurs` для русской clean speech;
- маленький synthetic noise pool: `fan`, `keyboard`, `traffic`, `cafe-babble-like`.

Это не финальный датасет. Он нужен, чтобы получить **первый обучающийся результат** и проверить весь pipeline в Colab.


In [ ]:
reset_paths(
    'data/colab_seed',
    'manifests/colab_clean.jsonl',
    'manifests/colab_noise.jsonl',
)
Path('manifests').mkdir(exist_ok=True)

run('uv run python scripts/prepare_first_colab_dataset.py --output-root data/colab_seed --num-clean 120 --seed 42')
run('uv run noise-suppression manifest build data/colab_seed/clean manifests/colab_clean.jsonl --kind clean --speaker-depth 0')
run('uv run noise-suppression manifest build data/colab_seed/noise manifests/colab_noise.jsonl --kind noise')
assert_nonempty_manifest('manifests/colab_clean.jsonl')
assert_nonempty_manifest('manifests/colab_noise.jsonl')
run('uv run noise-suppression manifest summarize manifests/colab_clean.jsonl')
run('uv run noise-suppression manifest summarize manifests/colab_noise.jsonl')


## 4. Создаем synthetic mixtures

Для первого Colab run берем умеренный размер:

- `1200` synthetic mixtures total;
- после split получится примерно `960 train / 240 val`.

Этого достаточно, чтобы увидеть, учится ли tiny model вообще.


In [ ]:
import json
from pathlib import Path

mix_config = {
    'seed': 42,
    'mixing': {
        'clean_manifest': '../manifests/colab_clean.jsonl',
        'noise_manifest': '../manifests/colab_noise.jsonl',
        'rir_manifest': None,
        'sample_rate': 16000,
        'num_examples': 1200,
        'min_duration_sec': 2.5,
        'max_duration_sec': 4.0,
        'snr_min_db': -2.0,
        'snr_max_db': 18.0,
        'focus_snr_min_db': 0.0,
        'focus_snr_max_db': 10.0,
        'focus_probability': 0.75,
        'reverb_probability': 0.0,
        'target_peak': 0.95,
    },
}

mix_config_path = Path('configs/colab_first_result.runtime.yaml')
mix_config_path.write_text(json.dumps(mix_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(mix_config_path)
print(mix_config_path.read_text(encoding='utf-8'))


In [ ]:
reset_paths(
    'manifests/colab_mix_plan.jsonl',
    'data/colab_rendered',
    'manifests/colab_train.jsonl',
    'manifests/colab_val.jsonl',
    'outputs/colab_identity_val',
)
Path('outputs').mkdir(exist_ok=True)

run('uv run noise-suppression mix plan configs/colab_first_result.runtime.yaml manifests/colab_mix_plan.jsonl')
assert_nonempty_manifest('manifests/colab_mix_plan.jsonl')
run('uv run noise-suppression mix render manifests/colab_mix_plan.jsonl data/colab_rendered --overwrite')
assert_nonempty_manifest('data/colab_rendered/rendered_colab_mix_plan.jsonl')
run('uv run noise-suppression manifest split data/colab_rendered/rendered_colab_mix_plan.jsonl manifests/colab_train.jsonl manifests/colab_val.jsonl --val-ratio 0.2 --seed 42')
assert_nonempty_manifest('manifests/colab_train.jsonl')
assert_nonempty_manifest('manifests/colab_val.jsonl')
run('uv run noise-suppression baseline apply manifests/colab_val.jsonl outputs/colab_identity_val --mode identity')
run('uv run noise-suppression metrics si-sdr manifests/colab_val.jsonl outputs/colab_identity_val')


## 5. Готовим train config с локальным сохранением модели

Критически важный момент:

- `output_dir` указывает на локальную папку `outputs/colab_first_result`;
- `checkpoint_mirror_dir` выключен и равен `None`;
- после **каждой эпохи** локально сохраняются `epoch_XXX.pt`, `last.pt`, `best.pt`, `history.json`, `resolved_config.json`.


In [ ]:
train_config = {
    'seed': 42,
    'data': {
        'train_manifest': '../manifests/colab_train.jsonl',
        'val_manifest': '../manifests/colab_val.jsonl',
        'sample_rate': 16000,
        'segment_seconds': 3.0,
        'batch_size': 8,
        'num_workers': 2,
        'limit_train': None,
        'limit_val': None,
    },
    'model': {
        'n_fft': 512,
        'hop_length': 128,
        'win_length': 512,
        'hidden_channels': 24,
    },
    'training': {
        'epochs': 6,
        'learning_rate': 0.001,
        'waveform_loss_weight': 1.0,
        'magnitude_loss_weight': 0.5,
        'device': 'auto',
        'output_dir': '../outputs/colab_first_result',
        'checkpoint_mirror_dir': None,
        'save_every_epoch': True,
        'log_interval': 20,
    },
}

train_config_path = Path('configs/colab_train.runtime.yaml')
train_config_path.write_text(json.dumps(train_config, ensure_ascii=False, indent=2), encoding='utf-8')
print(train_config_path)
print(train_config_path.read_text(encoding='utf-8'))


In [ ]:
reset_paths('outputs/colab_first_result')
run('uv run noise-suppression train fit configs/colab_train.runtime.yaml')


## 6. Смотрим историю обучения и checkpoint-ы


In [ ]:
import json
import matplotlib.pyplot as plt

runtime_output_dir = Path('outputs/colab_first_result')
history = json.loads((runtime_output_dir / 'history.json').read_text(encoding='utf-8'))['history']

print('Local checkpoints:')
for path in sorted(runtime_output_dir.glob('*')):
    print('  ', path.name)

required_checkpoints = [
    runtime_output_dir / 'best.pt',
    runtime_output_dir / 'last.pt',
    runtime_output_dir / 'history.json',
    runtime_output_dir / 'resolved_config.json',
]
missing = [str(path) for path in required_checkpoints if not path.exists()]
assert not missing, f'Не найдены обязательные checkpoint-файлы: {missing}'

epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_loss = [row['val_loss'] for row in history]
val_sisdr = [row['val_si_sdr'] for row in history]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, marker='o', label='train_loss')
plt.plot(epochs, val_loss, marker='o', label='val_loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Loss curves')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, val_sisdr, marker='o', color='tab:green', label='val_si_sdr')
plt.xlabel('epoch')
plt.ylabel('SI-SDR')
plt.title('Validation SI-SDR')
plt.grid(True)
plt.legend()
plt.show()

history


## 7. Валидируем `best.pt` на val subset


In [ ]:
reset_paths('outputs/colab_model_val')
run('uv run noise-suppression train infer outputs/colab_first_result/best.pt manifests/colab_val.jsonl outputs/colab_model_val')
run('uv run noise-suppression metrics si-sdr manifests/colab_val.jsonl outputs/colab_identity_val')
run('uv run noise-suppression metrics si-sdr manifests/colab_val.jsonl outputs/colab_model_val')


## 8. Слушаем один пример

Это полезно, чтобы проверить не только метрику, но и характер артефактов.


In [ ]:
from IPython.display import Audio, display
import json

val_rows = [json.loads(line) for line in Path('manifests/colab_val.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
sample = val_rows[0]
enhanced_path = Path('outputs/colab_model_val') / f"{sample['id']}.wav"

print('Sample id:', sample['id'])
print('Noisy:', sample['noisy_path'])
print('Clean:', sample['clean_path'])
print('Enhanced:', enhanced_path)

display(Audio(sample['noisy_path']))
display(Audio(sample['clean_path']))
display(Audio(str(enhanced_path)))


## 9. Что делать дальше

Если этот первый run прошел нормально, следующий разумный шаг:

1. увеличить число synthetic mixtures до `2000-4000`;
2. расширить noise pool;
3. добавить маленький real noisy eval set;
4. заменить `TinyMaskNet` на `FullSubNet-like` backbone;
5. сравнить результат с `DeepFilterNet`.

Checkpoint-ы лежат локально в `outputs/colab_first_result`.
